### Runnable Base Class:
- The Runnable class serves as the foundational building block. All other specialized `Runnables` inherit from this class.

In [1]:
from langchain_core.runnables import Runnable

class MyRunnable(Runnable):
    def invoke(self , input):
        return input.upper()

In [2]:
# creating an object of Our runnable
runnable = MyRunnable()
result = runnable.invoke("hello world")
print(f"[Output]: {result}")

[Output]: HELLO WORLD


### RunnableMap

Executes multiple Runnables in parallel and aggregates their results.
RunnableMap is used to run multiple operations in parallel on the same input and return 
all results as a dictionary.

Input <br>
  │   <br>
  ├── Operation A <br>
  ├── Operation B <br>
  └── Operation C <br>
        ↓<br>
   Results combined into a dictionary <br>

So the same input is sent to several runnables simultaneously, and each runnable produces 
its own output.

In [4]:
from langchain_core.runnables import RunnableMap 
runnable_map = RunnableMap({
    "uppercase": lambda x: x.upper(),
    "reverse": lambda x: x[: : -1]
})
result = runnable_map.invoke("langchain") # same input will be given to all the functions.
result

{'uppercase': 'LANGCHAIN', 'reverse': 'niahcgnal'}

**Why RunnableMap Exists?**
In LLM pipelines you often need multiple outputs from the same input.

**RunnableMap is used when you want to**:
- Apply multiple transformations
- To the same input
- Simultaneously
- And collect outputs in a structured dictionary

### RunnableSequence
Chains Runnables sequentially, passing the output of one as input to the next.
RunnableSequence is a sequential chain of runnables in LangChain that executues each
step one after another, passing the output of one step as the input to the next.

It is useful when we want to compose multiple runnables together in a structured workflow.

In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
# define the model
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "text-generation",
    temperature = 0.1,
    max_new_tokens = 512
)
model = ChatHuggingFace(llm = llm)

In [3]:
prompt = PromptTemplate(
    template = "Write a joke on this topic: {topic}",
    input_variables = ['topic']
)

In [4]:
str_parser = StrOutputParser()

In [9]:
runnable_seq = RunnableSequence(
    prompt , model , str_parser
)
output = runnable_seq.invoke({
    "topic": "machine learning"
})
print(f"Joke:\n{output}")

Joke:
Why did the machine learning algorithm refuse to go to the beach?

Because it heard the sand was too loose, and it didn't want to classify its toes!


### RunnableParallel
- RunnableParallel is a runnable that allows multiple runnables to execute in parallel.<br>
- Each runnable receives the same input and processes it independently, producing a dictionary of outputs.
- Althrough output are different but all the runnables get the same input.

In [7]:
prompt_points = PromptTemplate(
    template = "Generate 5 key points on the given topic: {topic}",
    input_variables = ['topic']
)
prompt_joke = PromptTemplate(
    template = "Write a joke on this topic: {topic}",
    input_variables = ['topic']
)

In [8]:
from langchain_core.runnables import RunnableParallel
parallel_runnable = RunnableParallel({
    "key points": RunnableSequence(prompt_points , model , str_parser),
    "Joke": RunnableSequence(prompt_joke , model , str_parser)
})

In [13]:
output = parallel_runnable.invoke({
    "topic": 'AI'
})
print(output)

{'key points': 'Certainly! Here are five key points on the topic of AI (Artificial Intelligence):\n\n1. **Definition and Scope**: AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. It encompasses a wide range of technologies and techniques, including machine learning, natural language processing, robotics, and expert systems.\n\n2. **Machine Learning**: A core component of AI, machine learning involves algorithms that allow computer systems to improve their performance on a specific task through experience. This learning process is often unsupervised, semi-supervised, or supervised, depending on the availability and nature of the training data.\n\n3. **Applications**: AI has a broad range of applications across various industries, including healthcare (diagnosis, drug discovery), finance (fraud detection, algorithmic trading), transpo

### RunnablePassthrough
RunnablePassthrough is a runnable is a special Runnable primitive that simply returns the
input as output without modifying it.

In [9]:
from langchain_core.runnables import RunnablePassthrough

In [10]:
prompt_joke = PromptTemplate(
    template = "Write a joke on this topic: {topic}",
    input_variables = ['topic']
)
prompt_joke_explaination = PromptTemplate(
    template = "Explain the joke: {joke}"
)

In [11]:
# make the chain to generate the joke
joke_gen = RunnableSequence(prompt_joke , model , str_parser)
chain = RunnableParallel({
    "Joke": RunnablePassthrough(),
    "Explaination": RunnableSequence(prompt_joke_explaination , model , str_parser)
})

# connect joke gen and chain
final_chain = RunnableSequence(joke_gen , chain)

In [17]:
output = final_chain.invoke({
    "topic": "AI"
})

In [18]:
output

{'Joke': 'Why did the AI cross the road?\n\nTo get to the algorithm on the other side!',
 'Explaination': 'This joke is a play on the classic joke "Why did the [animal] cross the road?" which is often used to set up a humorous punchline. In this case, the punchline is a reference to the core function of AI, which is to process and learn from data, much like an algorithm. The joke humorously personifies the AI as having a goal to reach a specific destination (the algorithm) just like a person might cross a road to reach a destination.'}

### Runnable Lambda
- RunnableLambda is a runnable primitive that allows you to apply custom python functions
within an AI pipeline.
- It acts like a middleware between different AI components, enabling preprocessing,
transformation, API calls, filtering and post-processing in a LangChain workflow.

In [5]:
import string
def word_counter(input: str) -> int:
    """Count the words in input string"""
    translator = str.maketrans('' , '' , string.punctuation)
    input = input.translate(translator)
    return len(input.split(" "))

word_counter("Hello! how are you? I am Tipto.")    

7

In [13]:
from langchain_core.runnables import RunnableLambda

In [14]:
joke_gen = RunnableSequence(prompt_joke , model , str_parser)
parallel_runnable = RunnableParallel({
    "Joke": RunnablePassthrough(),
    "Word counter": RunnableLambda(lambda x: word_counter(x)),
    "Joke explaination": RunnableSequence(prompt_joke_explaination , model , str_parser)
})

In [15]:
chain = RunnableSequence(joke_gen , parallel_runnable)
output = chain.invoke({
    "topic": "Machine learning"
})

In [16]:
output

{'Joke': 'Why did the machine learning algorithm go to therapy?\n\nBecause it had too many issues with overfitting and was struggling to generalize!',
 'Word counter': 21,
 'Joke explaination': 'This joke plays on the concept of overfitting in machine learning, which is a common issue where a model performs well on the training data but poorly on new, unseen data. The punchline humorously personifies the machine learning algorithm, suggesting it is seeking therapy to address its problems with overfitting. The algorithm is metaphorically "struggling to generalize," meaning it has difficulty applying what it has learned to new situations, a key challenge in machine learning.'}

### Runnable Branch
Runnable Branch is a control flow component in LangChain that allows you to conditionally 
route input data to different chains or runnables based on custom logic.

- It functions like a conditional block(eg..if/else if/else) for chains- where you define
a set of condition functions, each associated with a runnable(eg..LLM call, prompt chain 
or tool). The first matching condition is executed.
If not condition matches, a default runnable used(if provided[optional]). 

In [10]:
from langchain_core.runnables import RunnableBranch,RunnablePassthrough

In [12]:
prompt_report = PromptTemplate(
    template = "Write a report on the given topic:\n{topic}",
    input_variables = ['topic']
)
prompt_summarize = PromptTemplate(
    template = "Summarize the given report:\n{report}",
    input_variables = ['report']
)
prompt_explain = PromptTemplate(
    template = "Explain the given report:\n{report}",
    input_variables = ['report']
)
report_gen_chain = RunnableSequence(prompt_report , model , str_parser)
report = report_gen_chain.invoke({
    "topic": "machine learning"
})

runnable_branch = RunnableBranch(
    (lambda x: word_counter(x["report"]) >= 200 , RunnableSequence(prompt_summarize,model,str_parser)),
    (lambda x: word_counter(x['report']) < 100 , RunnableSequence(prompt_explain,model,str_parser)),
    RunnablePassthrough(report)
)

print(f"[INFO]: word count in the report: {word_counter(report)}")
output = runnable_branch.invoke({
    "report": report
})
print(f"[INFO]:\n{output}")

[INFO]: word count in the report: 300
[INFO]:
**Summary of the Report on Machine Learning**

This report provides an overview of machine learning (ML), its applications, key techniques, and future trends. ML involves developing algorithms and statistical models to enable computer systems to improve performance on specific tasks through experience.

**Overview:**
- ML is categorized into three main types: supervised learning, unsupervised learning, and reinforcement learning.
- The concept has roots in early computing but gained significant traction in the 1990s and 2000s with advancements in big data and computational power.

**Key Techniques:**
- **Supervised Learning**: Includes classification (e.g., spam detection) and regression (e.g., stock price prediction).
- **Unsupervised Learning**: Includes clustering (e.g., customer segmentation) and dimensionality reduction (e.g., PCA for image compression).
- **Reinforcement Learning**: Involves learning through trial and error (e.g., tra